TASK 1.1:

In [1]:
import csv
import json

In [2]:
# Creating the parse_row function to convert default strings to original datatypes:
def parse_row(row):
    parsed = dict(row)
    parsed['age'] = int(parsed['age'])
    parsed['prior_no_shows'] = int(parsed['prior_no_shows'])
    parsed['fee_inr'] = int(parsed['fee_inr'])
    parsed['distance_km'] = float(parsed['distance_km'])
    return parsed           

In [3]:
# Reading the file using CSV module 
with open("Datasets/appointments.csv",'r',newline = '') as file:
    reader = csv.DictReader(file)
    
    header = reader.fieldnames
    print("Header:",header)
    
    rows = list(reader)
    print("\nRows Count:",len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Rows Count: 1020


In [4]:
# Filtering out the rows with ValueError's and creating a new file for further usage:
accepted = []
rejected = []
for row in rows:
    try:
        parsed_row = parse_row(row)
        accepted.append(parsed_row)
    except ValueError:
        rejected.append(row)

In [5]:
# migrating the rejected list into JSON for clinic references:
with open("rejected_rows.json",'w') as out_f:
    json.dump(rejected,out_f,indent=2)

print(f'accepted :{len(accepted)} rows and rejected :{len(rejected)} rows')

accepted :988 rows and rejected :32 rows


TASK 1.2:




In [6]:
# Necessary libraries:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('Datasets/appointments.csv')
print('Shape :',df.shape)

Shape : (1020, 14)


In [8]:
print('Info :',df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB
Info : None


In [9]:
print('\n Missing Values :\n',df.isnull().sum())


 Missing Values :
 appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [10]:
print('Duplicates:',df.duplicated().sum())


Duplicates: 20


In [11]:
# Dropped duplicated rows and checking no of rows :
duplicates = df[df.duplicated()==True]
df = df.drop_duplicates()
df.shape

(1000, 14)

In [12]:
df['booking_channel'].unique()

<StringArray>
[   'Phone',  'Walk-in',      'App', ' walk-in',     'APP ',  'Walk-In',
   'Phone ',    'PHONE',      'app',     ' App',  'walk-in',    'phone',
 'Walk-in ',   ' phone',  'WALK-IN',     'App ']
Length: 16, dtype: str

In [13]:
# cleaning the data and convert the data types:
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
df['appt_date'] = pd.to_datetime(df['appt_date'],errors = 'coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'],errors = 'coerce')

In [14]:
df['booking_channel'].unique()

<StringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str

In [15]:

df = df.dropna(subset =['status','appt_date'])
df.shape

(990, 14)

Task 1.3:

In [16]:
# Set invalid age (< 1 or > 100) to missing (NaN)
df['age'] = df['age'].mask((df['age'] < 1) | (df['age'] > 100), np.nan)

# Fill missing age with median and missing distance_km with median 
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median).round().astype(int)

dist_median = df['distance_km'].median()
df['distance_km'] = df['distance_km'].fillna(dist_median)

print('Age median :',age_median)
print('Dist median :',dist_median)

Age median : 48.0
Dist median : 8.25


In [17]:
# Compute Q1, Q3, IQR and cap distance_km and fee_inr 

# 1. distance_km
dist_q1 = df['distance_km'].quantile(0.25)
dist_q3 = df['distance_km'].quantile(0.75)
dist_iqr = dist_q3 - dist_q1
dist_lower = dist_q1 - 1.5 * dist_iqr
dist_upper = dist_q3 + 1.5 * dist_iqr

# Count capped before applying the change
dist_capped_count = ((df['distance_km'] < dist_lower) | (df['distance_km'] > dist_upper)).sum()
df['distance_km'] = df['distance_km'].clip(lower=dist_lower, upper=dist_upper)

# 2. fee_inr
fee_q1 = df['fee_inr'].quantile(0.25)
fee_q3 = df['fee_inr'].quantile(0.75)
fee_iqr = fee_q3 - fee_q1
fee_lower = fee_q1 - 1.5 * fee_iqr
fee_upper = fee_q3 + 1.5 * fee_iqr

# Count capped before applying the change
fee_capped_count = ((df['fee_inr'] < fee_lower) | (df['fee_inr'] > fee_upper)).sum()
df['fee_inr'] = df['fee_inr'].clip(lower=fee_lower, upper=fee_upper)

# Round distance_km to 1 decimal & fee_inr to integer
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

# Print the results
print(f"Capped values in 'distance_km': {dist_capped_count}")
print(f"Capped values in 'fee_inr': {fee_capped_count}")

Capped values in 'distance_km': 26
Capped values in 'fee_inr': 3


Task 1.4:

In [18]:
# calculating lead days based on booking and appt days:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days

# mapping no and no show values:
df["no_show"] = df["status"].apply(lambda s: 1 if s == "No-Show" else 0)

# Extract X
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

#  Mean of each column
col_means = X.mean(axis=0)

# Number and share of rows with lead_days > 14
lead_days_col = X[:, 3]
mask_lead_14 = lead_days_col > 14
num_lead_14 = mask_lead_14.sum()
share_lead_14 = mask_lead_14.mean()

# Z-scores of the distance_km column
dist_col = X[:, 1]
dist_mean = np.mean(dist_col)
dist_std = np.std(dist_col)
# Avoid division by zero if std is 0
dist_z = (dist_col - dist_mean) / dist_std 
num_outlier_z = (abs(dist_z) > 2).sum()

# Print calculations
print("Column Means (age, distance_km, fee_inr, lead_days):")
print(col_means)
print(f"\nLead days > 14 - Count: {num_lead_14}, Share: {share_lead_14:.2%}")
print(f"Distance km outliers (|z| > 2) count: {num_outlier_z}")

Column Means (age, distance_km, fee_inr, lead_days):
[ 48.17373737   9.50090909 635.71414141  11.81919192]

Lead days > 14 - Count: 296, Share: 29.90%
Distance km outliers (|z| > 2) count: 53


Task 1.5:

In [19]:
import requests
df1 = pd.read_csv('datasets/clinics.csv')
df1.head()

,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [20]:
url = 'https://api.open-meteo.com/v1/forecast'

temp_list = []

for idx, row in df1.iterrows():
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }
    
    temp_c = None
    success = False
    
    # Attempt 1
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if "hourly" in data and "temperature_2m" in data["hourly"]:
                temp_c = data["hourly"]["temperature_2m"][0]
                success = True
            else:
                print(f"Failure for {row['clinic_name']}: 'hourly' or 'temperature_2m' is missing in response.")
        else:
            print(f"Failure for {row['clinic_name']}: Status code {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Attempt 1 failed for {row['clinic_name']} with RequestException: {e}")

    # Retry once if the first attempt failed
    if not success:
        print(f"Retrying once for {row['clinic_name']}...")
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    success = True
                else:
                    print(f"Retry failure for {row['clinic_name']}: 'hourly' or 'temperature_2m' missing.")
            else:
                print(f"Retry failure for {row['clinic_name']}: Status code {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"Retry failed for {row['clinic_name']} with RequestException: {e}")
            
    temp_list.append(temp_c)
df1['temp_c'] = temp_list
df1.to_csv("clinics_enriched.csv", index=False)

In [25]:
df1

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,14.4
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,15.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,22.3
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,15.3
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,15.3
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,24.2
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,15.2
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,17.0


Task 6:

In [21]:
# converting date columns to str formats for further SQL usage:
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df = df[['appointment_id', 'patient_id', 'clinic_id', 'booked_on','appt_date','age','gender','distance_km','fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days','no_show']]

In [22]:
# checking for null values and id's are unique:
assert df["appointment_id"].nunique() == len(df), "appointment_id is not unique!"
assert df.isnull().sum().sum() == 0, "Missing values found in clean dataset!"

In [23]:
# creating new file with cleaned data:
df.to_csv('appointments_clean.csv',index = False)

In [24]:
# Extracting a new SQL commands file :
with open('load_data.sql', 'w', newline='') as f:
    for _, r in df1.iterrows():
        temp_val = 'NULL' if pd.isna(r['temp_c']) else r['temp_c']
        f.write(
            f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}',"
            f"'{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp_val});\n"
        )
    
    for _, r in df.iterrows():
        f.write(
        "INSERT INTO appointments VALUES"
        f" ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});\n"
    )